### 🔥 Deep Learning Cave — Chapter 7: The Distillation Forge

> *"A master doesn't just teach the answer. They teach the uncertainty behind it."*

You've descended to the Distillation Forge. Here, knowledge is not mined raw — it is **compressed, refined, and poured** from a giant model into a small one.

**Chapter 7** is the secret behind **Gemma 2**: how a 2B model can think like a 27B model. Not magic — **Knowledge Distillation**.

---

#### 🗺️ Your Position in the Cave

```
[■■■■■■■■■■] Chapter 1: PyTorch Foundations ✓
[■■■■■■■■■■] Chapter 2: The Original Transformer ✓
[■■■■■■■■■■] Chapter 3: Modern LLaMA ✓
[■■■■■■■■■■] Chapter 4: Vision Transformer ✓
[■■■■■■■■■■] Chapter 5: I-JEPA ✓
[■■■■■■■■■■] Chapter 6: Mixture of Experts ✓
[■■□□□□□□□□] Chapter 7: Knowledge Distillation ← You are here
 ├── Hinton's temperature softmax
 ├── On-Policy Distillation (Gemma 2)
 └── Logit Soft-Capping

[□□□□□□□□□□] Chapter 8: Coming soon...
```

**Paper**: *"Gemma 2: Improving Open Language Models at a Practical Size"* (Google DeepMind, 2024) — [arXiv:2408.00118](https://arxiv.org/abs/2408.00118)

---

#### The Big Idea

**Knowledge Distillation (KD)** is a training technique where a small **student** model learns from a large **teacher** model — not just from ground-truth labels, but from the teacher's **full probability distribution** over outputs.

Why does this matter? Hard labels say: `cat = 1.0, dog = 0.0, car = 0.0`  
Teacher soft labels say: `cat = 0.70, dog = 0.25, car = 0.05` ← **rich information!**

The teacher's uncertainty encodes **similarity structure** between classes — information that hard labels throw away.

```
┌─────────────────────────────────────────────────────────────────┐
│              Knowledge Distillation Architecture                │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   Input Data                                                    │
│       │                                                         │
│       ├──────────────────┐                                      │
│       │                  │                                      │
│       ▼                  ▼                                      │
│  ┌──────────┐      ┌──────────┐                                 │
│  │  TEACHER │      │ STUDENT  │                                 │
│  │ (Large)  │      │ (Small)  │                                 │
│  │  27B 🐘  │      │  2B 🐭   │                                 │
│  └────┬─────┘      └────┬─────┘                                 │
│       │                  │                                      │
│       ▼                  ▼                                      │
│  ┌──────────┐      ┌──────────┐                                 │
│  │  Soft    │      │  Soft    │                                 │
│  │  Logits  │      │  Logits  │                                 │
│  └────┬─────┘      └────┬─────┘                                 │
│       │                  │                                      │
│       └──────────┬───────┘                                      │
│                  │                                              │
│                  ▼                                              │
│           ┌────────────┐                                        │
│           │ KL Diverge │  ← Student learns teacher's            │
│           │   Loss     │    uncertainty distribution            │
│           └────────────┘                                        │
│                                                                 │
│   Total Loss = α * CE(student, labels) + β * KL(student, teacher)│
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

#### Gemma 2's Breakthrough

Gemma 2 didn't just use vanilla KD. It combined:
1. **Knowledge Distillation** from a 27B teacher → trains 2B and 9B models
2. **On-Policy Distillation** — student generates its own completions, teacher scores them
3. **Logit Soft-Capping** — bounds logits with `tanh` for training stability
4. **50× token budget** — training far beyond compute-optimal with distillation gradients

Result: A 9B model that outperforms many 27B+ models. A 2B model that's competitive with 7B models.

---

#### Learning Path

| Part | Component | What You'll Learn |
|------|-----------|------------------|
| 1 | Setup & Config | KD hyperparameters |
| 2 | Soft Labels | Why hard labels waste information |
| 3 | Temperature Softmax | Hinton's key insight |
| 4 | KL Divergence Loss | The distillation objective |
| 5 | Teacher-Student Models | Architecture setup |
| 6 | Logit Soft-Capping | Gemma 2's stability trick |
| 7 | On-Policy Distillation | Gemma 2's advanced method |
| 8 | Data Pipeline | CIFAR-10 setup |
| 9 | Training Loop | Full KD training |
| 10 | Evaluation | KD vs. baseline comparison |

---

### Part 1: Setup and Configuration

🎯 **What it does**: Define teacher, student, and distillation hyperparameters

🔧 **Why it matters**: KD has two critical knobs — the **temperature** (how soft the teacher's distribution is) and **alpha** (balance between soft and hard losses).

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dataclasses import dataclass
from typing import Optional

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
@dataclass
class DistillationConfig:
    # Dataset
    n_classes: int = 10          # CIFAR-10
    img_size: int = 32
    in_channels: int = 3

    # Teacher model (larger)
    teacher_channels: list = None  # [64, 128, 256, 512]
    teacher_fc_dim: int = 512

    # Student model (smaller — ~8x fewer params)
    student_channels: list = None  # [32, 64, 128]
    student_fc_dim: int = 256

    # Distillation hyperparameters
    temperature: float = 4.0     # T > 1 softens distributions (Hinton's trick)
    alpha: float = 0.7           # Weight for soft (KD) loss
    # (1 - alpha) weight for hard (CE) loss

    # Gemma 2 specific
    soft_cap: float = 30.0       # Logit soft-capping value

    # Training
    epochs: int = 30
    batch_size: int = 128
    teacher_lr: float = 1e-3
    student_lr: float = 1e-3
    weight_decay: float = 1e-4

    def __post_init__(self):
        if self.teacher_channels is None:
            self.teacher_channels = [64, 128, 256, 512]
        if self.student_channels is None:
            self.student_channels = [32, 64, 128]

config = DistillationConfig()

print("=" * 55)
print("  Knowledge Distillation Config (Gemma 2 inspired)")
print("=" * 55)
print(f"  Temperature (T):    {config.temperature}   ← softens teacher")
print(f"  Alpha (α):          {config.alpha}   ← KD loss weight")
print(f"  1 - Alpha:          {1-config.alpha}   ← CE loss weight")
print(f"  Soft-cap value:     {config.soft_cap}  ← Gemma 2 logit bound")
print(f"  Teacher channels:   {config.teacher_channels}")
print(f"  Student channels:   {config.student_channels}")
print("=" * 55)

---

### Part 2: Hard Labels vs. Soft Labels

🎯 **What it does**: Demonstrate why hard labels throw away information

🔧 **Why it matters**: This is the fundamental insight behind KD. A teacher's uncertainty isn't weakness — it's **knowledge**.

In [ ]:
# Visualize the difference between hard and soft labels
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# Hard label (one-hot): ground truth is "cat"
hard_label = torch.zeros(10)
hard_label[3] = 1.0  # cat = index 3

# Soft label from a teacher model (what a smart 27B model might output)
# The teacher knows: cats look like dogs and deer, nothing like trucks
soft_label = torch.tensor([
    0.005,  # airplane
    0.003,  # automobile
    0.020,  # bird    ← has fur too!
    0.700,  # cat     ← correct answer
    0.080,  # deer    ← similar pose
    0.150,  # dog     ← very similar!
    0.010,  # frog
    0.025,  # horse
    0.002,  # ship
    0.005,  # truck
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hard label plot
bars0 = axes[0].bar(classes, hard_label.numpy(), color='#e74c3c', alpha=0.8)
axes[0].set_title('Hard Label (One-Hot)\n"cat=1, everything else=0"', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Probability')
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis='x', rotation=45)
axes[0].text(0.5, 0.85, '❌ Throws away similarity info!\nDog and cat are equally wrong.', 
             ha='center', transform=axes[0].transAxes,
             bbox=dict(boxstyle='round', facecolor='#fadbd8', alpha=0.9), fontsize=10)

# Soft label plot
colors = ['#3498db' if c not in ['cat', 'dog', 'deer'] else '#e74c3c' if c == 'cat' else '#e67e22'
          for c in classes]
bars1 = axes[1].bar(classes, soft_label.numpy(), color=colors, alpha=0.8)
axes[1].set_title('Soft Label (Teacher Distribution)\n"cat=0.70, dog=0.15, deer=0.08..."', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Probability')
axes[1].set_ylim(0, 0.85)
axes[1].tick_params(axis='x', rotation=45)
axes[1].text(0.5, 0.85, '✅ Rich inter-class relationships!\nDog is similar, ship is not.', 
             ha='center', transform=axes[1].transAxes,
             bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.9), fontsize=10)

plt.tight_layout()
plt.suptitle('The Core Insight: Soft Labels Encode Knowledge', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("\nInformation content (entropy):")
hard_entropy = -(hard_label * (hard_label + 1e-8).log()).sum().item()
soft_entropy = -(soft_label * (soft_label + 1e-8).log()).sum().item()
print(f"  Hard label entropy: {hard_entropy:.4f} bits  ← almost zero information")
print(f"  Soft label entropy: {soft_entropy:.4f} bits  ← rich similarity structure")

💡 **Key Insight**: A hard label says "cat is cat". A teacher's soft label says "cat looks a lot like a dog, somewhat like a deer, nothing like a truck." That **inter-class similarity structure** is the teacher's dark knowledge — and it's what the student absorbs during distillation.

---

### Part 3: Temperature Softmax — Hinton's Key Insight

🎯 **What it does**: Soften logit distributions using a temperature parameter T

🔧 **Why it matters**: Without temperature, the teacher's distribution is too peaky (essentially one-hot after softmax). Temperature `T > 1` reveals the **dark knowledge** hiding in the small logit values.

In [ ]:
def softmax_with_temperature(logits: torch.Tensor, T: float) -> torch.Tensor:
    """Apply softmax with temperature scaling.
    
    Higher T → softer (more uniform) distribution
    Lower T → harder (more peaked) distribution
    T = 1.0 → standard softmax
    T → ∞  → uniform distribution
    T → 0  → one-hot (argmax)
    """
    return F.softmax(logits / T, dim=-1)


# Simulate teacher logits for a "cat" image
# Note: logits are raw (pre-softmax) scores from the model
teacher_logits = torch.tensor([
    -2.1,  # airplane
    -3.0,  # automobile
    -1.2,  # bird
     3.5,  # cat     ← highest
     0.3,  # deer
     1.8,  # dog     ← second highest
    -2.5,  # frog
     0.1,  # horse
    -3.2,  # ship
    -2.8,  # truck
])

temperatures = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, T in enumerate(temperatures):
    probs = softmax_with_temperature(teacher_logits, T)
    entropy = -(probs * probs.log()).sum().item()

    color = '#e74c3c' if T == 1.0 else '#3498db'
    axes[i].bar(range(10), probs.numpy(), color=color, alpha=0.75)
    axes[i].set_title(f'T = {T}  (entropy={entropy:.2f})', fontweight='bold')
    axes[i].set_xticks(range(10))
    axes[i].set_xticklabels([c[:4] for c in classes], rotation=45, fontsize=8)
    axes[i].set_ylim(0, 1.05)

    if T < 1.0:
        axes[i].text(0.5, 0.75, '← Almost one-hot\n  (T < 1)', ha='center',
                     transform=axes[i].transAxes,
                     bbox=dict(boxstyle='round', facecolor='#fadbd8', alpha=0.85), fontsize=9)
    elif T == 4.0:
        axes[i].text(0.5, 0.75, '← Gemma 2 uses\n  T ≈ 4 here', ha='center',
                     transform=axes[i].transAxes,
                     bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.85), fontsize=9)
    elif T >= 16.0:
        axes[i].text(0.5, 0.75, '← Near uniform\n  (T → ∞)', ha='center',
                     transform=axes[i].transAxes,
                     bbox=dict(boxstyle='round', facecolor='#fdebd0', alpha=0.85), fontsize=9)

plt.suptitle('Effect of Temperature T on Softmax Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Temperature effect summary:")
for T in temperatures:
    probs = softmax_with_temperature(teacher_logits, T)
    entropy = -(probs * probs.log()).sum().item()
    max_prob = probs.max().item()
    print(f"  T={T:5.1f}: max_prob={max_prob:.3f}, entropy={entropy:.3f}")

💡 **Key Insight**: Hinton et al. (2015) discovered that at T=1, the small logits (e.g., dog=1.8) carry meaningful information but get suppressed in the softmax. Raising T amplifies those small probabilities, revealing the **dark knowledge** — the teacher's implicit understanding of class similarity.

Gemma 2 uses distillation at token level: the teacher's distribution over the **entire vocabulary** (e.g., 256K tokens) provides enormously rich gradients compared to one-hot next-token targets.

---

### Part 4: The Distillation Loss

🎯 **What it does**: Define the combined loss: KL divergence (soft) + cross-entropy (hard)

🔧 **Why it matters**: The student minimizes two objectives simultaneously — match the teacher's soft distribution AND predict the correct hard label.

In [ ]:
def distillation_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    labels: torch.Tensor,
    T: float,
    alpha: float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Combined Knowledge Distillation loss.

    Loss = alpha * T² * KL(student_soft || teacher_soft)
         + (1 - alpha) * CE(student_logits, hard_labels)

    Note: T² rescaling ensures gradients are same magnitude
          regardless of temperature (Hinton et al., 2015).

    Args:
        student_logits: (B, C) raw logits from student
        teacher_logits: (B, C) raw logits from teacher (detached)
        labels: (B,) ground truth class indices
        T: temperature (softens distributions)
        alpha: weight for soft (KD) loss

    Returns:
        total_loss, soft_loss, hard_loss
    """
    # Soft loss: KL divergence between student and teacher at temperature T
    student_soft = F.log_softmax(student_logits / T, dim=-1)
    teacher_soft = F.softmax(teacher_logits.detach() / T, dim=-1)

    # KL(P_teacher || P_student) = sum(P_teacher * log(P_teacher / P_student))
    # F.kl_div expects log-probabilities for input
    soft_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean')
    soft_loss = soft_loss * (T ** 2)  # T² rescaling

    # Hard loss: standard cross-entropy with ground truth labels
    hard_loss = F.cross_entropy(student_logits, labels)

    # Combined loss
    total_loss = alpha * soft_loss + (1 - alpha) * hard_loss

    return total_loss, soft_loss, hard_loss


# Demonstrate loss behavior
B, C = 4, 10
torch.manual_seed(42)

teacher_logits_demo = torch.randn(B, C) * 3
student_logits_demo = torch.randn(B, C)  # untrained student
labels_demo = torch.randint(0, C, (B,))

total, soft, hard = distillation_loss(
    student_logits_demo, teacher_logits_demo, labels_demo,
    T=config.temperature, alpha=config.alpha
)

print("Distillation Loss Breakdown:")
print(f"  Soft Loss (KL × T²):  {soft.item():.4f}  ← match teacher distribution")
print(f"  Hard Loss (CE):        {hard.item():.4f}  ← match ground truth labels")
print(f"  Total Loss:            {total.item():.4f}")
print(f"  = {config.alpha} × {soft.item():.4f} + {1-config.alpha} × {hard.item():.4f}")
print(f"  = {config.alpha * soft.item():.4f} + {(1-config.alpha) * hard.item():.4f}")

💡 **Key Insight**: The `T²` scaling factor is critical — without it, the magnitude of gradients from the soft loss would shrink as T increases (since dividing logits by T reduces their magnitudes). The `T²` correction keeps the gradients at the right scale.

**Gemma 2 extension**: Instead of classification CE, the hard loss becomes **next-token prediction CE** over the full vocabulary. The soft loss becomes **KL divergence over 256K vocabulary tokens** — providing enormously richer learning signal than any image classification task.

---

### Part 5: Teacher and Student Architectures

🎯 **What it does**: Build CNN teacher (large) and student (small) models

🔧 **Why it matters**: The size gap between teacher and student determines how much "compression" happens. Gemma 2: 27B teacher → 2B student (13× compression).

In [ ]:
class ConvBlock(nn.Module):
    """Conv → BN → ReLU → (optional) MaxPool."""

    def __init__(self, in_channels, out_channels, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class TeacherCNN(nn.Module):
    """Large teacher model for CIFAR-10.

    Architecture: 4 ConvBlocks → GlobalAvgPool → 2-layer FC head
    Think of this as our humble 27B 'teacher' analog.
    """

    def __init__(self, config: DistillationConfig):
        super().__init__()
        channels = config.teacher_channels  # [64, 128, 256, 512]
        self.features = nn.Sequential(
            ConvBlock(config.in_channels, channels[0]),    # 32→16
            ConvBlock(channels[0], channels[1]),           # 16→8
            ConvBlock(channels[1], channels[2]),           # 8→4
            ConvBlock(channels[2], channels[3], pool=False) # 4→4 (no pool)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels[3], config.teacher_fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(config.teacher_fc_dim, config.n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


class StudentCNN(nn.Module):
    """Small student model for CIFAR-10.

    Architecture: 3 ConvBlocks → GlobalAvgPool → 1-layer FC head
    Think of this as our humble 2B 'student' analog.
    """

    def __init__(self, config: DistillationConfig):
        super().__init__()
        channels = config.student_channels  # [32, 64, 128]
        self.features = nn.Sequential(
            ConvBlock(config.in_channels, channels[0]),    # 32→16
            ConvBlock(channels[0], channels[1]),           # 16→8
            ConvBlock(channels[1], channels[2]),           # 8→4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels[2], config.student_fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(config.student_fc_dim, config.n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


# Instantiate and count parameters
teacher = TeacherCNN(config).to(device)
student = StudentCNN(config).to(device)
student_baseline = StudentCNN(config).to(device)  # same arch, trained without KD

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print("Model size comparison:")
print(f"  Teacher params: {teacher_params:>10,}  (the 27B 🐘)")
print(f"  Student params: {student_params:>10,}  (the 2B  🐭)")
print(f"  Compression:    {teacher_params / student_params:.1f}×")
print()

# Quick sanity check
dummy = torch.randn(4, 3, 32, 32).to(device)
teacher_out = teacher(dummy)
student_out = student(dummy)
print(f"Teacher output shape: {teacher_out.shape}")
print(f"Student output shape: {student_out.shape}")

---

### Part 6: Logit Soft-Capping (Gemma 2's Stability Innovation)

🎯 **What it does**: Bound logits to a fixed range using a differentiable tanh cap

🔧 **Why it matters**: When training on massive token counts with distillation, logits can explode. Soft-capping prevents numerical instability without sharp clipping gradients.

In [ ]:
def logit_soft_cap(logits: torch.Tensor, soft_cap: float) -> torch.Tensor:
    """Apply Gemma 2's logit soft-capping.

    Formula: logits ← soft_cap × tanh(logits / soft_cap)

    Properties:
    - Differentiable everywhere (unlike hard clipping)
    - Bounded output: (-soft_cap, +soft_cap)
    - Linear near 0 (doesn't distort small logits)
    - Saturates gracefully for large values

    Gemma 2 uses:
    - Attention logits: soft_cap = 50.0
    - Final output logits: soft_cap = 30.0
    """
    return soft_cap * torch.tanh(logits / soft_cap)


# Visualize soft-cap vs. hard clip vs. no cap
x = torch.linspace(-60, 60, 500)

no_cap = x
hard_clip = torch.clamp(x, -30, 30)
soft_cap_30 = logit_soft_cap(x, 30.0)
soft_cap_50 = logit_soft_cap(x, 50.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Function comparison
axes[0].plot(x.numpy(), no_cap.numpy(), 'gray', linewidth=1.5, linestyle='--', label='No cap (identity)', alpha=0.6)
axes[0].plot(x.numpy(), hard_clip.numpy(), 'r', linewidth=2, label='Hard clip (±30)', alpha=0.8)
axes[0].plot(x.numpy(), soft_cap_30.numpy(), 'b', linewidth=2.5, label='Soft cap (cap=30) ← final logits')
axes[0].plot(x.numpy(), soft_cap_50.numpy(), 'g', linewidth=2.5, label='Soft cap (cap=50) ← attention logits')
axes[0].set_xlabel('Input Logit')
axes[0].set_ylabel('Output Logit')
axes[0].set_title('Logit Soft-Capping vs. Hard Clip', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].axhline(30, color='b', linestyle=':', alpha=0.4)
axes[0].axhline(-30, color='b', linestyle=':', alpha=0.4)
axes[0].grid(True, alpha=0.3)

# Right: Gradient comparison
x_grad = torch.linspace(-60, 60, 500).requires_grad_(True)

def compute_grad(func_out):
    func_out.sum().backward(retain_graph=True)
    grad = x_grad.grad.detach().clone()
    x_grad.grad = None
    return grad

grad_soft_30 = compute_grad(logit_soft_cap(x_grad, 30.0))
grad_soft_50 = compute_grad(logit_soft_cap(x_grad, 50.0))

axes[1].plot(x_grad.detach().numpy(), grad_soft_30.numpy(), 'b', linewidth=2.5, label='Soft cap (cap=30) gradient')
axes[1].plot(x_grad.detach().numpy(), grad_soft_50.numpy(), 'g', linewidth=2.5, label='Soft cap (cap=50) gradient')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Input Logit')
axes[1].set_ylabel('Gradient (d output / d input)')
axes[1].set_title('Gradients — Smooth at All Values!', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].text(0, 0.6, '← Gradient gracefully\n   decays to 0 at extremes\n   (no dead zeros like clip)', 
             ha='center',
             bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.9), fontsize=9)

plt.suptitle('Gemma 2: Logit Soft-Capping for Training Stability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Demo the formula
sample_logits = torch.tensor([-100.0, -30.0, -10.0, 0.0, 10.0, 30.0, 100.0])
capped = logit_soft_cap(sample_logits, 30.0)
print("Soft-capping demo (cap=30):")
for raw, cap in zip(sample_logits.tolist(), capped.tolist()):
    print(f"  {raw:7.1f}  →  {cap:6.3f}")

💡 **Key Insight**: Gemma 2 applies soft-capping in **two places**:
1. **Attention logits** (cap=50): prevents attention scores from becoming extremely confident, improving generalization
2. **Output logits** (cap=30): prevents the final vocabulary distribution from becoming too peaked, aiding knowledge distillation

The formula `cap × tanh(x / cap)` is elegant: it's linear near zero (preserving small differences) but saturates smoothly at ±cap (preventing explosions). Unlike `torch.clamp`, gradients don't go to zero — they gracefully approach zero.

---

### Part 7: On-Policy Distillation (Gemma 2's Advanced Method)

🎯 **What it does**: The student generates its own outputs; the teacher evaluates them

🔧 **Why it matters**: Standard KD trains on the teacher's outputs given ground-truth inputs. On-policy KD trains on the **student's own generations** — fixing the train-inference mismatch.

In [ ]:
def on_policy_distillation_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    T: float = 1.0
) -> torch.Tensor:
    """On-Policy Distillation loss (Gemma 2 approach).

    In language models:
    1. Student generates completions autoregressively (on-policy)
    2. Teacher computes logits over student's completions
    3. Student minimizes KL(teacher || student) over its own outputs

    Key difference from standard KD:
    - Standard KD: teacher/student see same ground-truth tokens
    - On-Policy KD: student sees its OWN tokens (as in inference)

    This eliminates 'exposure bias' — the mismatch between
    train-time (ground truth input) and inference-time (model output).

    For classification demo: we simulate this by adding noise
    to inputs (simulating student's 'own distribution').
    """
    student_log_probs = F.log_softmax(student_logits / T, dim=-1)
    teacher_probs = F.softmax(teacher_logits.detach() / T, dim=-1)

    # KL(teacher || student) — teacher is the target distribution
    kl_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
    return kl_loss * (T ** 2)


# Visualize: Standard KD vs On-Policy KD training flow
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def draw_flow(ax, title, steps, colors):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(steps) + 1)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

    for i, (step, color) in enumerate(zip(steps, colors)):
        y = len(steps) - i
        rect = mpatches.FancyBboxPatch((0.5, y - 0.35), 9, 0.7,
                                        boxstyle='round,pad=0.1',
                                        facecolor=color, edgecolor='#2c3e50',
                                        linewidth=1.5)
        ax.add_patch(rect)
        ax.text(5, y, step, ha='center', va='center', fontsize=9.5,
                fontweight='bold', color='#2c3e50')
        if i < len(steps) - 1:
            ax.annotate('', xy=(5, y - 0.4), xytext=(5, y - 0.6),
                        arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=2))

standard_steps = [
    "1. Ground-truth tokens: 'The cat sat'",
    "2. Teacher sees 'The cat sat' → produces soft logits",
    "3. Student sees 'The cat sat' → produces logits",
    "4. KL(teacher_logits || student_logits)",
    "⚠️  Train: real tokens | Inference: student tokens → MISMATCH!",
]

on_policy_steps = [
    "1. Ground-truth prompt: 'The cat'",
    "2. Student autoregressively generates: 'sat on'",
    "3. Teacher evaluates student's output: 'sat on'",
    "4. KL(teacher_logits || student_logits) on student tokens",
    "✅  Train: student tokens | Inference: student tokens → MATCH!",
]

std_colors = ['#d6eaf8', '#d6eaf8', '#d6eaf8', '#d6eaf8', '#fadbd8']
onp_colors = ['#d6eaf8', '#d5f5e3', '#d5f5e3', '#d6eaf8', '#d5f5e3']

draw_flow(axes[0], 'Standard KD (offline)', standard_steps, std_colors)
draw_flow(axes[1], 'On-Policy KD (Gemma 2)', on_policy_steps, onp_colors)

plt.suptitle('Standard vs. On-Policy Distillation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("On-Policy KD advantages (from Gemma 2 paper):")
print("  1. Eliminates exposure bias (train-inference distribution mismatch)")
print("  2. Student learns to recover from its OWN mistakes")
print("  3. More sample-efficient: 50× token training becomes practical")
print("  4. Teacher acts as a dense reward signal at every token position")

💡 **Key Insight**: Standard KD has **exposure bias** — the student trains on ground-truth tokens but at inference time sees its own (potentially wrong) tokens. On-policy KD closes this gap: the student generates its own text, the teacher scores every token, and the KL divergence minimization happens in the student's own distribution.

Gemma 2 trained the 2B and 9B models on **50× compute-optimal tokens** (trillions of tokens) using this approach. Without on-policy distillation, this would cause severe exposure bias.

---

### Part 8: Data Pipeline

🎯 **What it does**: Load CIFAR-10 with standard augmentations

🔧 **Why it matters**: Both teacher and student train on the same data — the key difference is the loss function, not the data.

In [ ]:
from pathlib import Path

data_path = './data'

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar_extracted = Path(data_path) / 'cifar-10-batches-py'
cifar_archive = Path(data_path) / 'cifar-10-python.tar.gz'

if not cifar_extracted.exists() and not cifar_archive.exists():
    print("❌ CIFAR-10 not found. Automatic download is blocked (network restrictions detected).")
    print("\n✅ MANUAL DOWNLOAD STEPS:")
    print("   1. Visit: https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz")
    print(f"   2. Save to: {Path(data_path).absolute()}/cifar-10-python.tar.gz")
    print("   3. Re-run this cell")
    raise RuntimeError("Manual download required.")

train_dataset = torchvision.datasets.CIFAR10(
    root=data_path, train=True, download=False, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root=data_path, train=False, download=False, transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=0, pin_memory=True)

print(f"✅ CIFAR-10 loaded")
print(f"   Train samples: {len(train_dataset):,} | Batches: {len(train_loader)}")
print(f"   Test samples:  {len(test_dataset):,} | Batches: {len(test_loader)}")

---

### Part 9: Training Loop — Teacher First, Then Student

🎯 **What it does**: Train teacher normally → freeze it → train student with KD loss

🔧 **Why it matters**: The teacher must be pre-trained to high accuracy before distillation. A bad teacher produces bad soft labels — garbage in, garbage out.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model accuracy on a dataset."""
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total


def train_standard(model, loader, optimizer, epoch, model_name='Model'):
    """Standard training with cross-entropy (no distillation)."""
    model.train()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), 100.0 * correct / total


def train_with_distillation(student, teacher, loader, optimizer, T, alpha, use_soft_cap=True):
    """Train student with KD loss from frozen teacher.

    Includes optional Gemma 2 logit soft-capping.
    """
    student.train()
    teacher.eval()
    total_loss = soft_loss_sum = hard_loss_sum = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Student forward
        student_logits = student(images)

        # Teacher forward (no grad — teacher is frozen!)
        with torch.no_grad():
            teacher_logits = teacher(images)

        # Optional: Gemma 2 logit soft-capping
        if use_soft_cap:
            student_logits_capped = logit_soft_cap(student_logits, config.soft_cap)
            teacher_logits_capped = logit_soft_cap(teacher_logits, config.soft_cap)
        else:
            student_logits_capped = student_logits
            teacher_logits_capped = teacher_logits

        # Distillation loss
        loss, soft_loss, hard_loss = distillation_loss(
            student_logits_capped, teacher_logits_capped, labels, T=T, alpha=alpha
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        soft_loss_sum += soft_loss.item()
        hard_loss_sum += hard_loss.item()
        correct += (student_logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    n = len(loader)
    return total_loss / n, soft_loss_sum / n, hard_loss_sum / n, 100.0 * correct / total

print("Training utilities defined.")
print(f"  Standard training: cross-entropy only")
print(f"  KD training: α={config.alpha} × KL + {1-config.alpha} × CE at T={config.temperature}")

In [ ]:
# ─────────────────────────────────────────────────────────
# Phase 1: Train the TEACHER (standard cross-entropy)
# ─────────────────────────────────────────────────────────
print("Phase 1: Training Teacher Model 🐘")
print("-" * 55)

teacher_optimizer = torch.optim.Adam(
    teacher.parameters(), lr=config.teacher_lr, weight_decay=config.weight_decay
)
teacher_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    teacher_optimizer, T_max=config.epochs
)

teacher_train_accs, teacher_test_accs = [], []

for epoch in range(config.epochs):
    tr_loss, tr_acc = train_standard(teacher, train_loader, teacher_optimizer, epoch, 'Teacher')
    teacher_scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        test_acc = evaluate(teacher, test_loader)
        teacher_train_accs.append(tr_acc)
        teacher_test_accs.append(test_acc)
        print(f"  Epoch {epoch+1:2d}/{config.epochs} | Loss: {tr_loss:.3f} | "
              f"Train: {tr_acc:.1f}% | Test: {test_acc:.1f}%")

teacher_final_acc = evaluate(teacher, test_loader)
print(f"\n✅ Teacher trained | Final test acc: {teacher_final_acc:.2f}%")

# Freeze teacher — it's done
for param in teacher.parameters():
    param.requires_grad = False
teacher.eval()
print("   Teacher frozen — acting as oracle for student training")

In [ ]:
# ─────────────────────────────────────────────────────────
# Phase 2a: Train STUDENT with Knowledge Distillation
# ─────────────────────────────────────────────────────────
print("Phase 2a: Training Student with KD 🐭 + 📚")
print("-" * 55)

student_optimizer = torch.optim.Adam(
    student.parameters(), lr=config.student_lr, weight_decay=config.weight_decay
)
student_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    student_optimizer, T_max=config.epochs
)

kd_test_accs, kd_soft_losses, kd_hard_losses = [], [], []

for epoch in range(config.epochs):
    total_loss, soft_loss, hard_loss, tr_acc = train_with_distillation(
        student, teacher, train_loader, student_optimizer,
        T=config.temperature, alpha=config.alpha, use_soft_cap=True
    )
    student_scheduler.step()
    kd_soft_losses.append(soft_loss)
    kd_hard_losses.append(hard_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        test_acc = evaluate(student, test_loader)
        kd_test_accs.append(test_acc)
        print(f"  Epoch {epoch+1:2d}/{config.epochs} | "
              f"Soft: {soft_loss:.3f} | Hard: {hard_loss:.3f} | "
              f"Train: {tr_acc:.1f}% | Test: {test_acc:.1f}%")

student_kd_acc = evaluate(student, test_loader)
print(f"\n✅ Student (KD) | Final test acc: {student_kd_acc:.2f}%")

In [ ]:
# ─────────────────────────────────────────────────────────
# Phase 2b: Train BASELINE student (no KD, just CE)
# ─────────────────────────────────────────────────────────
print("Phase 2b: Training Baseline Student (no KD) 🐭")
print("-" * 55)

baseline_optimizer = torch.optim.Adam(
    student_baseline.parameters(), lr=config.student_lr, weight_decay=config.weight_decay
)
baseline_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    baseline_optimizer, T_max=config.epochs
)

baseline_test_accs = []

for epoch in range(config.epochs):
    tr_loss, tr_acc = train_standard(student_baseline, train_loader, baseline_optimizer, epoch)
    baseline_scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        test_acc = evaluate(student_baseline, test_loader)
        baseline_test_accs.append(test_acc)
        print(f"  Epoch {epoch+1:2d}/{config.epochs} | Loss: {tr_loss:.3f} | "
              f"Train: {tr_acc:.1f}% | Test: {test_acc:.1f}%")

baseline_acc = evaluate(student_baseline, test_loader)
print(f"\n✅ Baseline Student | Final test acc: {baseline_acc:.2f}%")

---

### Part 10: Evaluation and Visualization

🎯 **What it does**: Compare teacher, student-KD, and baseline student performance

🔧 **Why it matters**: The gap between student-KD and baseline student directly measures the value of knowledge distillation.

In [ ]:
# ── Accuracy curves ──────────────────────────────────────
epochs_logged = list(range(1, config.epochs + 1, 5)) + [config.epochs]
epochs_logged = sorted(set(epochs_logged))[:len(kd_test_accs)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy comparison
axes[0].plot(epochs_logged, kd_test_accs, 'b-o', linewidth=2.5, markersize=6,
             label=f'Student + KD (T={config.temperature}, α={config.alpha})')
axes[0].plot(epochs_logged, baseline_test_accs, 'r--s', linewidth=2, markersize=6,
             label='Student (no KD — baseline)')
axes[0].axhline(teacher_final_acc, color='g', linewidth=2, linestyle=':',
                label=f'Teacher ceiling ({teacher_final_acc:.1f}%)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Knowledge Distillation Benefit', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: Loss components during KD training
axes[1].plot(kd_soft_losses, 'b', linewidth=2, label='Soft Loss (KL × T²) — match teacher')
axes[1].plot(kd_hard_losses, 'r', linewidth=2, linestyle='--', label='Hard Loss (CE) — match labels')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('KD Loss Decomposition During Training', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
teacher_p = sum(p.numel() for p in teacher.parameters())
student_p = sum(p.numel() for p in student.parameters())

kd_gain = student_kd_acc - baseline_acc
teacher_gap = teacher_final_acc - student_kd_acc

print("\n" + "=" * 65)
print("  RESULTS SUMMARY")
print("=" * 65)
print(f"  {'Model':<30} {'Params':>10}  {'Test Acc':>10}")
print("-" * 65)
print(f"  {'Teacher (large)':<30} {teacher_p:>10,}  {teacher_final_acc:>9.2f}%")
print(f"  {'Student + KD (ours)':<30} {student_p:>10,}  {student_kd_acc:>9.2f}%")
print(f"  {'Student (no KD, baseline)':<30} {student_p:>10,}  {baseline_acc:>9.2f}%")
print("=" * 65)
print(f"  KD gain over baseline:  +{kd_gain:.2f}%")
print(f"  Gap to teacher ceiling: -{teacher_gap:.2f}%")
print(f"  Model size reduction:   {teacher_p/student_p:.1f}×")
print("=" * 65)

In [ ]:
# ── Ablation: effect of temperature on KD performance ────
print("Quick ablation: KD loss @ different temperatures")
print("(Using current trained teacher & student logits)\n")

# Collect a batch of logits
teacher.eval()
with torch.no_grad():
    sample_images, sample_labels = next(iter(test_loader))
    sample_images = sample_images.to(device)
    t_logits = teacher(sample_images)
    s_logits = student(sample_images)

temps = [0.5, 1, 2, 4, 8, 16]
kl_losses = []

for T in temps:
    t_soft = F.softmax(t_logits / T, dim=-1)
    s_soft = F.log_softmax(s_logits / T, dim=-1)
    kl = F.kl_div(s_soft, t_soft, reduction='batchmean').item() * (T ** 2)
    kl_losses.append(kl)
    print(f"  T={T:5.1f}: KL × T² = {kl:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(temps, kl_losses, 'b-o', linewidth=2.5, markersize=8)
plt.axvline(config.temperature, color='r', linestyle='--', alpha=0.7, label=f'Chosen T={config.temperature}')
plt.xlabel('Temperature T')
plt.ylabel('KL Divergence × T²')
plt.title('Distillation Loss vs Temperature', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

### Summary

#### What We Built

A complete **Knowledge Distillation** pipeline inspired by Gemma 2:

| Component | Purpose | Gemma 2 Analog |
|-----------|---------|----------------|
| **Temperature Softmax** | Reveal teacher's dark knowledge | Applied over 256K vocab |
| **KL Divergence Loss** | Match teacher's distribution | Token-level distillation |
| **Logit Soft-Capping** | Stabilize training with tanh | cap=50 (attn), cap=30 (logits) |
| **On-Policy Distillation** | Eliminate exposure bias | Student generates, teacher scores |
| **T² Rescaling** | Normalize gradient magnitude | Compensates for temperature |

#### Why Knowledge Distillation is Gemma 2's Breakthrough

| Aspect | Without KD | With KD (Gemma 2) |
|--------|-----------|------------------|
| Training signal | One-hot next token | Full distribution over 256K tokens |
| Gradient richness | Sparse | Dense |
| Token budget | Compute-optimal | 50× over-training becomes viable |
| 9B vs. 27B | 9B ≪ 27B | 9B ≈ 27B (on many benchmarks!) |

#### The Hinton Equations

```
# 1. Temperature softmax (soften teacher)
p_teacher = softmax(logits_teacher / T)
p_student = softmax(logits_student / T)

# 2. Soft loss (match teacher distribution)
L_soft = T² × KL(p_teacher || p_student)

# 3. Hard loss (match ground truth)
L_hard = CrossEntropy(logits_student, labels)

# 4. Combined loss
L_total = α × L_soft + (1 - α) × L_hard

# 5. Gemma 2 bonus: logit soft-capping
logits = soft_cap × tanh(logits / soft_cap)
```

#### What's Next?

From here, you could explore:
1. **Feature distillation**: Match intermediate layer activations (not just outputs)
2. **Self-distillation**: Teacher and student are the same architecture across epochs
3. **Speculative decoding**: Use student as a fast draft model, teacher as verifier
4. **RLHF + KD**: Combine KD with human preference learning

---

🎉 **You've unlocked the secret behind Gemma 2!** A 9B model beating 27B models isn't magic — it's a brilliantly executed pipeline of dark knowledge transfer, exposure bias elimination, and logit stability tricks. Now you know exactly how it works. 🔥

---

### 🗿 Chapter 7 Complete: The Forge Master

You descended into the Distillation Forge expecting alchemy.

What you found was **engineering** — beautiful, precise, and brutally effective.

---

#### 🌟 The Full Cave Map

| Chamber | Architecture | Core Insight |
|---------|-------------|--------------|
| **1** | PyTorch | Tensors are everything |
| **2** | Transformer | Attention is all you need |
| **3** | LLaMA | Modern LLMs improve the basics |
| **4** | ViT | Vision is just sequences |
| **5** | I-JEPA | Predict representations, not pixels |
| **6** | MoE | Not every expert needs every input |
| **7** | KD + Gemma 2 | **A great teacher makes small models mighty** |

---

#### 🧠 The Deepest Lesson

Gemma 2's insight isn't about scale. It's about **information density**.

A one-hot label `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]` wastes 9/10 of the gradient signal.  
A teacher's soft label `[.005, .003, .02, .70, .08, .15, ...]` uses **every position**.

Now multiply this across 256,000 vocabulary tokens, 2 trillion training steps, and on-policy generation — and you understand why a 2B model can think like a 27B model.

**Dark knowledge is just information we forgot to use.**

---

> *"The best teacher is not the one who knows the most, but the one who knows how to pass it on."*  
> — Geoffrey Hinton, probably (he definitely implied it in the 2015 paper)

---

🔥 **The forge is yours. Keep building.** 🔥

📖 *[Gemma 2 Paper: arXiv:2408.00118](https://arxiv.org/abs/2408.00118)*  
📖 *[Original KD Paper: Hinton et al. (2015)](https://arxiv.org/abs/1503.02531)*